# FER2013 Enhanced Emotion Classification

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Dataset, khảo sát dữ liệu và chuẩn bị dữ liệu đầu vào.

Notebook này dùng trên Google Colab để thực hiện pipeline huấn luyện chính của project nhận diện cảm xúc khuôn mặt. Các nhãn **Phụ trách/Phối hợp** trong từng cell đã được ghi theo tên thành viên để mỗi bạn dễ theo dõi phần việc của mình khi chạy lại notebook.

| Thành viên | MSSV | Phần chính trong notebook |
|---|---:|---|
| Đặng Hoàng Quân | 20235813 | Dataset, khảo sát dữ liệu, preprocessing, xử lý mất cân bằng, augmentation và `tf.data` pipeline. |
| Nguyễn Minh Quân | 20235816 | Trưởng nhóm; thiết kế Custom CNN 4 block, cấu hình optimizer/loss/callbacks và huấn luyện model. |
| Đinh Văn Phạm Việt | 20235870 | Đánh giá model trên test set, classification report, confusion matrix, phân tích confidence và lỗi sai. |
| Đinh Hữu Nhật Minh | 20235778 | Tích hợp inference/webcam, so sánh Haar/MTCNN và kiểm chứng mô hình trên ảnh thực tế. |

Các bước chính trong notebook:

1. **Step 1:** Chọn và tải dataset `abhilash88/fer2013-enhanced`.
2. **Step 2:** Khảo sát dataset: split, cột dữ liệu, label mapping, phân bố lớp, kích thước ảnh và metadata chất lượng.
3. **Step 3:** Tiền xử lý dữ liệu thành tensor `48x48x1` để chuẩn bị cho CNN.
4. **Step 4:** Xử lý mất cân bằng lớp và tạo augmentation preview.
5. **Step 5:** Đóng gói dữ liệu thành `tf.data.Dataset` cho train/validation/test.
6. **Step 6:** Xây dựng kiến trúc Custom CNN 4 block.
7. **Step 7:** Huấn luyện model, lưu checkpoint tốt nhất và training curves.
8. **Step 8:** Đánh giá test set, xuất metric, confusion matrix và ví dụ lỗi sai.


## 0. Cài thư viện

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Chuẩn bị môi trường để tải và khảo sát dataset.

Colab thường đã có TensorFlow. Cell này cài các thư viện cần cho dataset và báo cáo khảo sát dữ liệu.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Cài thư viện phục vụ tải dataset và khảo sát dữ liệu.
!pip -q install datasets scikit-learn matplotlib seaborn pillow


## 1. Kiểm tra môi trường

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Kiểm tra môi trường chạy và tạo thư mục output.

Cell này kiểm tra TensorFlow/GPU và tạo các thư mục chuẩn để lưu kết quả khảo sát.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Kiểm tra môi trường Colab và tạo thư mục output.
import json
from pathlib import Path

import numpy as np
import tensorflow as tf
from datasets import load_dataset

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

Path("models").mkdir(exist_ok=True)
Path("reports").mkdir(exist_ok=True)
Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)


## Step 1 - Chọn và tải dataset FER2013-enhanced

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Dataset selection và dataset loading.

Nhóm chọn `abhilash88/fer2013-enhanced` vì:

- Phù hợp trực tiếp với bài toán Facial Emotion Recognition.
- Ảnh đã ở dạng khuôn mặt grayscale kích thước nhỏ `48x48`, phù hợp CNN nhẹ.
- Có sẵn `train`, `validation`, `test`, thuận tiện cho train/evaluate.
- Có 7 cảm xúc đúng với mục tiêu project: `angry`, `disgust`, `fear`, `happy`, `sad`, `surprise`, `neutral`.
- Có metadata chất lượng ảnh như `quality_score`, `brightness`, `contrast`, `focus_score`, `sample_weight` để phân tích thêm.

Lưu ý khi báo cáo: dataset đã cung cấp ảnh và split sẵn, nhưng nhóm vẫn tự khảo sát, chuẩn hóa đầu vào, xử lý mất cân bằng, augmentation, train CNN và evaluate ở các bước sau.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Khai báo dataset, split kỳ vọng và các cột quan trọng.
DATASET_NAME = "abhilash88/fer2013-enhanced"
EXPECTED_SPLITS = ("train", "validation", "test")
IMAGE_COLUMNS = ("image", "pixels")
LABEL_COLUMNS = ("emotion", "emotion_name")

print("Dataset:", DATASET_NAME)
print("Expected splits:", EXPECTED_SPLITS)


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Tải dataset từ Hugging Face.
dataset = load_dataset(DATASET_NAME)
dataset


### 1.1. Xác nhận split và cột dữ liệu

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Kiểm tra dataset có đúng cấu trúc cần cho project không.

Cell này kiểm tra:

- Dataset có đủ `train`, `validation`, `test` không.
- Mỗi split có bao nhiêu dòng.
- Cột nào là ảnh.
- Cột nào là nhãn.
- Dataset có các metadata nào để phục vụ phân tích.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Tóm tắt split, cột ảnh, cột nhãn và lưu dataset summary.
def find_first_existing(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


summary = {
    "dataset_name": DATASET_NAME,
    "splits": {},
}

for split_name, split_data in dataset.items():
    columns = list(split_data.column_names)
    image_column = find_first_existing(columns, IMAGE_COLUMNS)
    label_column = find_first_existing(columns, LABEL_COLUMNS)
    summary["splits"][split_name] = {
        "num_rows": len(split_data),
        "columns": columns,
        "image_column": image_column,
        "label_column": label_column,
    }

missing_splits = [split for split in EXPECTED_SPLITS if split not in summary["splits"]]
if missing_splits:
    raise ValueError(f"Missing expected split(s): {missing_splits}")

for split_name, split_summary in summary["splits"].items():
    if split_summary["image_column"] is None:
        raise ValueError(f"Split '{split_name}' has no image column. Columns: {split_summary['columns']}")
    if split_summary["label_column"] is None:
        raise ValueError(f"Split '{split_name}' has no label column. Columns: {split_summary['columns']}")

summary_path = Path("reports/dataset_load_summary.json")
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Loaded dataset: {DATASET_NAME}")
for split_name in EXPECTED_SPLITS:
    split_summary = summary["splits"][split_name]
    print(
        f"- {split_name}: {split_summary['num_rows']} rows, "
        f"image='{split_summary['image_column']}', label='{split_summary['label_column']}'"
    )
print(f"Saved summary to: {summary_path}")


### 1.2. Xem thử một mẫu dữ liệu

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Kiểm tra trực quan một sample đầu tiên.

Mục tiêu là xác nhận một dòng dữ liệu có ảnh, nhãn và metadata đúng như kỳ vọng.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Xem một sample để xác nhận ảnh và nhãn.
sample = dataset["train"][0]
print("Sample keys:", sample.keys())

image_column = summary["splits"]["train"]["image_column"]
label_column = summary["splits"]["train"]["label_column"]

print("Label id:", sample[label_column])
print("Emotion name:", sample.get("emotion_name"))
print("Image type:", type(sample[image_column]))
sample[image_column]


## Step 2 - Khảo sát dữ liệu

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Data analysis và báo cáo dữ liệu.

Step này tạo các bằng chứng để báo cáo rằng nhóm hiểu dataset:

- Label mapping 7 cảm xúc.
- Phân bố class theo từng split.
- Kích thước và mode ảnh.
- Ảnh mẫu theo từng class.
- Metadata chất lượng ảnh.
- Nhận xét về mất cân bằng lớp.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Import thư viện phục vụ khảo sát dữ liệu.
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

sns.set_theme(style="whitegrid")

CANONICAL_LABELS = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "sad",
    "surprise",
    "neutral",
]


def normalize_name(name):
    return str(name).strip().lower().replace(" ", "_")


### 2.1. Xác định label mapping

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Xác nhận mapping giữa id và tên cảm xúc.

Output này được lưu vào `models/labels.json` để dùng lại khi train, evaluate và realtime demo.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Tạo label mapping 0..6 cho 7 cảm xúc.
label_mapping = {}

for row in dataset["train"]:
    label_id = int(row["emotion"])
    label_name = normalize_name(row.get("emotion_name", label_id))
    label_mapping[label_id] = label_name
    if len(label_mapping) == 7:
        break

label_mapping = dict(sorted(label_mapping.items()))
print(label_mapping)

missing_labels = [label for label in CANONICAL_LABELS if label not in label_mapping.values()]
if missing_labels:
    raise ValueError(f"Missing expected emotion labels: {missing_labels}")

Path("models/labels.json").write_text(
    json.dumps({str(k): v for k, v in label_mapping.items()}, indent=2),
    encoding="utf-8",
)
print("Saved label mapping to: models/labels.json")


### 2.2. Đếm số lượng class theo split

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Phân tích mất cân bằng lớp.

Phần này giúp chỉ ra class nào nhiều/ít. Đây là cơ sở cho bước sau dùng `sample_weight` hoặc `class_weight`.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Đếm số ảnh theo class ở train/validation/test.
class_counts = {}

for split_name in EXPECTED_SPLITS:
    counts = Counter(int(row["emotion"]) for row in dataset[split_name])
    class_counts[split_name] = {
        label_mapping[label_id]: counts.get(label_id, 0)
        for label_id in sorted(label_mapping)
    }

print(json.dumps(class_counts, indent=2))

Path("reports/class_distribution.json").write_text(json.dumps(class_counts, indent=2), encoding="utf-8")
print("Saved class counts to: reports/class_distribution.json")


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Vẽ biểu đồ phân bố class theo split.
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, split_name in zip(axes, EXPECTED_SPLITS):
    labels = list(class_counts[split_name].keys())
    values = list(class_counts[split_name].values())
    sns.barplot(x=labels, y=values, ax=ax, color="#4c78a8")
    ax.set_title(split_name)
    ax.set_xlabel("Emotion")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.savefig("reports/class_distribution.png", dpi=160, bbox_inches="tight")
plt.show()
print("Saved chart to: reports/class_distribution.png")


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Nhận xét nhanh về mất cân bằng lớp.
train_counts = class_counts["train"]
majority_class = max(train_counts, key=train_counts.get)
minority_class = min(train_counts, key=train_counts.get)
imbalance_ratio = train_counts[majority_class] / max(train_counts[minority_class], 1)

imbalance_summary = {
    "majority_class": majority_class,
    "majority_count": train_counts[majority_class],
    "minority_class": minority_class,
    "minority_count": train_counts[minority_class],
    "imbalance_ratio": round(float(imbalance_ratio), 2),
    "note": "Dataset is imbalanced; later training should consider sample_weight or class_weight.",
}

print(json.dumps(imbalance_summary, indent=2))
Path("reports/class_imbalance_summary.json").write_text(
    json.dumps(imbalance_summary, indent=2),
    encoding="utf-8",
)
print("Saved imbalance summary to: reports/class_imbalance_summary.json")


### 2.3. Kiểm tra kích thước ảnh và định dạng ảnh

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Xác nhận input ảnh trước khi preprocessing.

Dataset được kỳ vọng là ảnh grayscale `48x48`. Cell này kiểm tra trên tối đa 500 ảnh mỗi split để xác nhận nhanh.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Hàm chuyển dữ liệu ảnh về PIL Image để thống kê.
def to_pil_image(value):
    if isinstance(value, Image.Image):
        return value

    array = np.array(value)
    if array.ndim == 1:
        side = int(np.sqrt(array.size))
        array = array.reshape(side, side)
    return Image.fromarray(array.astype("uint8"))


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Kiểm tra size và mode ảnh trên mỗi split.
image_stats = {}

for split_name in EXPECTED_SPLITS:
    sizes = Counter()
    modes = Counter()
    split_data = dataset[split_name]
    image_column = summary["splits"][split_name]["image_column"]

    for row in split_data.select(range(min(500, len(split_data)))):
        image = to_pil_image(row[image_column])
        sizes[image.size] += 1
        modes[image.mode] += 1

    image_stats[split_name] = {
        "sizes": {str(k): v for k, v in sizes.items()},
        "modes": dict(modes),
    }

print(json.dumps(image_stats, indent=2))
Path("reports/image_stats.json").write_text(json.dumps(image_stats, indent=2), encoding="utf-8")
print("Saved image stats to: reports/image_stats.json")


### 2.4. Xem ảnh mẫu theo từng class

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Kiểm tra trực quan dữ liệu từng cảm xúc.

Ảnh mẫu giúp báo cáo dễ hiểu hơn và giúp phát hiện sớm nếu label/dữ liệu có vấn đề.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Lấy một ảnh mẫu cho mỗi class từ train set.
examples_by_class = {}

for row in dataset["train"]:
    label_id = int(row["emotion"])
    if label_id not in examples_by_class:
        examples_by_class[label_id] = row
    if len(examples_by_class) == len(label_mapping):
        break

fig, axes = plt.subplots(1, len(label_mapping), figsize=(15, 3))
for ax, label_id in zip(axes, sorted(label_mapping)):
    row = examples_by_class[label_id]
    image = to_pil_image(row["image"])
    ax.imshow(image, cmap="gray")
    ax.set_title(label_mapping[label_id])
    ax.axis("off")

plt.tight_layout()
plt.savefig("reports/sample_images_by_class.png", dpi=160, bbox_inches="tight")
plt.show()
print("Saved sample images to: reports/sample_images_by_class.png")


### 2.5. Khảo sát metadata chất lượng ảnh

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Phân tích metadata chất lượng ảnh.

Các cột như `brightness`, `contrast`, `quality_score`, `focus_score` giúp nhóm giải thích vì sao ảnh webcam thực tế có thể làm model khó dự đoán hơn.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Tính min/mean/median/max cho các cột chất lượng ảnh.
quality_columns = [
    column
    for column in ["quality_score", "brightness", "contrast", "focus_score", "brightness_score"]
    if column in dataset["train"].column_names
]

quality_summary = defaultdict(dict)

for split_name in EXPECTED_SPLITS:
    for column in quality_columns:
        values = np.array([row[column] for row in dataset[split_name]], dtype="float32")
        quality_summary[split_name][column] = {
            "min": float(np.min(values)),
            "mean": float(np.mean(values)),
            "median": float(np.median(values)),
            "max": float(np.max(values)),
        }

print(json.dumps(quality_summary, indent=2))
Path("reports/quality_summary.json").write_text(json.dumps(quality_summary, indent=2), encoding="utf-8")
print("Saved quality summary to: reports/quality_summary.json")


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Vẽ histogram cho metadata chất lượng ảnh của train set.
if quality_columns:
    fig, axes = plt.subplots(len(quality_columns), 1, figsize=(10, 3 * len(quality_columns)))
    if len(quality_columns) == 1:
        axes = [axes]

    for ax, column in zip(axes, quality_columns):
        values = np.array([row[column] for row in dataset["train"]], dtype="float32")
        sns.histplot(values, bins=40, ax=ax, color="#59a14f")
        ax.set_title(f"Train {column}")
        ax.set_xlabel(column)

    plt.tight_layout()
    plt.savefig("reports/quality_histograms.png", dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved quality histograms to: reports/quality_histograms.png")
else:
    print("No quality columns found.")


## Kết luận Step 1-2

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Tổng hợp nhận xét để đưa vào báo cáo/slide.

Sau Step 1-2, nhóm cần có các kết luận sau:

- Dataset `FER2013-enhanced` có đủ `train`, `validation`, `test`.
- Dataset có 35.887 ảnh: 25.117 train, 5.380 validation, 5.390 test.
- Ảnh đúng format `48x48` grayscale, mode `L`.
- Có 7 lớp cảm xúc đúng mục tiêu project.
- Dataset mất cân bằng rõ rệt: `happy` nhiều nhất, `disgust` ít nhất.
- Có metadata chất lượng ảnh để phân tích thêm.
- Các bước sau cần xử lý mất cân bằng lớp và chuẩn hóa tensor đầu vào trước khi train CNN.


## Step 3 - Tiền xử lý dữ liệu cho CNN

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Preprocessing dữ liệu đầu vào cho CNN.

Mục tiêu của Step 3 là chuyển dữ liệu từ dạng ảnh/label của Hugging Face Dataset thành tensor số đúng format cho CNN:

- Ảnh grayscale `48x48` -> tensor `48x48x1`.
- Pixel `[0, 255]` -> `[0.0, 1.0]`.
- Nhãn cảm xúc -> số nguyên `0..6`.
- Extract `sample_weight` để dùng ở bước xử lý mất cân bằng/training sau.

Step này **chưa làm augmentation**, **chưa tạo `tf.data.Dataset`**, và **chưa train model**.


### 3.1. Cấu hình preprocessing

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Định nghĩa input/output chuẩn cho CNN.

Cell này lưu cấu hình preprocessing vào `reports/preprocessing_config.json` để báo cáo và các bước sau dùng lại.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Cấu hình input/output chuẩn cho CNN.
IMG_SIZE = 48
NUM_CHANNELS = 1
INPUT_SHAPE = (IMG_SIZE, IMG_SIZE, NUM_CHANNELS)
NUM_CLASSES = len(label_mapping)
PIXEL_RANGE = (0.0, 1.0)

preprocessing_config = {
    "img_size": IMG_SIZE,
    "num_channels": NUM_CHANNELS,
    "input_shape": list(INPUT_SHAPE),
    "num_classes": NUM_CLASSES,
    "pixel_range": list(PIXEL_RANGE),
    "x_dtype": "float32",
    "y_dtype": "int64",
    "label_format": "integer class id",
    "normalization": "pixel / 255.0",
}

Path("reports/preprocessing_config.json").write_text(
    json.dumps(preprocessing_config, indent=2),
    encoding="utf-8",
)

print(json.dumps(preprocessing_config, indent=2))
print("Saved preprocessing config to: reports/preprocessing_config.json")


### 3.2. Hàm chuyển ảnh sang tensor

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Viết hàm xử lý ảnh đầu vào.

Mỗi ảnh được đưa về grayscale, resize chắc chắn về `48x48`, ép kiểu `float32`, normalize về `[0,1]`, rồi thêm channel dimension để thành `48x48x1`.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Hàm chuyển một ảnh thành tensor 48x48x1.
def preprocess_image(value):
    image = to_pil_image(value).convert("L")
    image = image.resize((IMG_SIZE, IMG_SIZE))

    array = np.asarray(image, dtype="float32") / 255.0
    array = np.expand_dims(array, axis=-1)
    return array.astype("float32")


# Kiểm tra nhanh trên một sample.
processed_sample = preprocess_image(dataset["train"][0][summary["splits"]["train"]["image_column"]])
print("Processed sample shape:", processed_sample.shape)
print("Processed sample dtype:", processed_sample.dtype)
print("Processed sample range:", float(processed_sample.min()), float(processed_sample.max()))


### 3.3. Chuyển từng split thành NumPy arrays

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Convert `train`, `validation`, `test` thành arrays.

Cell này tạo các biến trong RAM Colab:

- `x_train`, `y_train`, `w_train`
- `x_val`, `y_val`, `w_val`
- `x_test`, `y_test`, `w_test`

Trong đó `w_*` là `sample_weight`. Nếu dataset không có cột này, notebook sẽ dùng toàn bộ weight bằng `1.0`.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Convert một split thành x/y/sample_weight arrays.
USE_SAMPLE_WEIGHT = "sample_weight" in dataset["train"].column_names


def split_to_arrays(split_name):
    split_data = dataset[split_name]
    image_column = summary["splits"][split_name]["image_column"]
    label_column = summary["splits"][split_name]["label_column"]
    num_rows = len(split_data)

    images = np.empty((num_rows, IMG_SIZE, IMG_SIZE, NUM_CHANNELS), dtype="float32")
    labels = np.empty((num_rows,), dtype="int64")
    weights = np.ones((num_rows,), dtype="float32")

    for index, row in enumerate(split_data):
        images[index] = preprocess_image(row[image_column])
        labels[index] = int(row[label_column])
        if USE_SAMPLE_WEIGHT:
            weights[index] = float(row["sample_weight"])

    return images, labels, weights


x_train, y_train, w_train = split_to_arrays("train")
x_val, y_val, w_val = split_to_arrays("validation")
x_test, y_test, w_test = split_to_arrays("test")

print("Using sample_weight:", USE_SAMPLE_WEIGHT)
print("Train:", x_train.shape, y_train.shape, w_train.shape)
print("Validation:", x_val.shape, y_val.shape, w_val.shape)
print("Test:", x_test.shape, y_test.shape, w_test.shape)


### 3.4. Validate shape, dtype và pixel range

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Kiểm tra output preprocessing có đúng format không.

Nếu shape, dtype hoặc pixel range sai, cell này sẽ báo lỗi ngay để tránh đưa dữ liệu sai vào bước train.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Validate shape, dtype, pixel range và số lượng sample.
expected_shapes = {
    "train": (summary["splits"]["train"]["num_rows"], IMG_SIZE, IMG_SIZE, NUM_CHANNELS),
    "validation": (summary["splits"]["validation"]["num_rows"], IMG_SIZE, IMG_SIZE, NUM_CHANNELS),
    "test": (summary["splits"]["test"]["num_rows"], IMG_SIZE, IMG_SIZE, NUM_CHANNELS),
}

actual_arrays = {
    "train": (x_train, y_train, w_train),
    "validation": (x_val, y_val, w_val),
    "test": (x_test, y_test, w_test),
}

for split_name, (x_values, y_values, weights) in actual_arrays.items():
    expected_shape = expected_shapes[split_name]
    if x_values.shape != expected_shape:
        raise ValueError(f"{split_name} x shape mismatch: {x_values.shape} != {expected_shape}")
    if y_values.shape != (expected_shape[0],):
        raise ValueError(f"{split_name} y shape mismatch: {y_values.shape}")
    if weights.shape != (expected_shape[0],):
        raise ValueError(f"{split_name} weight shape mismatch: {weights.shape}")
    if x_values.dtype != np.float32:
        raise TypeError(f"{split_name} x dtype mismatch: {x_values.dtype}")
    if y_values.dtype != np.int64:
        raise TypeError(f"{split_name} y dtype mismatch: {y_values.dtype}")
    if float(x_values.min()) < 0.0 or float(x_values.max()) > 1.0:
        raise ValueError(f"{split_name} pixel range is outside [0, 1]")

print("All shape/dtype/range checks passed.")
print("x_train dtype:", x_train.dtype)
print("y_train dtype:", y_train.dtype)
print("x_train range:", float(x_train.min()), float(x_train.max()))


### 3.5. Validate label sau preprocessing

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Kiểm tra label không bị lệch hoặc mất class.

Cell này xác nhận label vẫn nằm trong `0..6` và phân bố label sau preprocessing khớp với Step 2.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Validate label id và phân bố label sau preprocessing.
preprocessed_class_counts = {}
expected_label_ids = np.array(sorted(label_mapping.keys()))

for split_name, (_, y_values, _) in actual_arrays.items():
    unique_labels = np.unique(y_values)
    if not np.array_equal(unique_labels, expected_label_ids):
        raise ValueError(f"{split_name} labels mismatch: {unique_labels} != {expected_label_ids}")

    counts = Counter(int(label_id) for label_id in y_values)
    preprocessed_class_counts[split_name] = {
        label_mapping[label_id]: counts.get(label_id, 0)
        for label_id in sorted(label_mapping)
    }

    if preprocessed_class_counts[split_name] != class_counts[split_name]:
        raise ValueError(f"{split_name} class distribution changed after preprocessing")

print(json.dumps(preprocessed_class_counts, indent=2))
print("All label checks passed.")


### 3.6. Xem ảnh sau preprocessing

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Kiểm tra trực quan ảnh sau normalize/reshape.

Cell này lưu ảnh preview vào `reports/preprocessed_samples.png`. Nếu ảnh bị méo, bị đảo màu hoặc không còn nhìn được khuôn mặt, cần sửa preprocessing trước khi train.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Hiển thị ảnh sau preprocessing để kiểm tra trực quan.
num_preview = min(8, len(x_train))
fig, axes = plt.subplots(1, num_preview, figsize=(2.2 * num_preview, 3))
if num_preview == 1:
    axes = [axes]

for index, ax in enumerate(axes):
    ax.imshow(x_train[index].squeeze(), cmap="gray", vmin=0.0, vmax=1.0)
    ax.set_title(label_mapping[int(y_train[index])])
    ax.axis("off")

plt.tight_layout()
plt.savefig("reports/preprocessed_samples.png", dpi=160, bbox_inches="tight")
plt.show()
print("Saved preprocessed samples to: reports/preprocessed_samples.png")


### 3.7. Lưu preprocessing summary

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Lưu kết quả kiểm tra preprocessing để đưa vào báo cáo.

File `reports/preprocessing_summary.json` tóm tắt shape, dtype, pixel range, label range và việc có dùng `sample_weight` hay không.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Lưu summary của Step 3.
preprocessing_summary = {
    "train_shape": list(x_train.shape),
    "validation_shape": list(x_val.shape),
    "test_shape": list(x_test.shape),
    "x_dtype": str(x_train.dtype),
    "y_dtype": str(y_train.dtype),
    "weight_dtype": str(w_train.dtype),
    "pixel_min": float(min(x_train.min(), x_val.min(), x_test.min())),
    "pixel_max": float(max(x_train.max(), x_val.max(), x_test.max())),
    "label_min": int(min(y_train.min(), y_val.min(), y_test.min())),
    "label_max": int(max(y_train.max(), y_val.max(), y_test.max())),
    "num_classes": NUM_CLASSES,
    "uses_sample_weight": bool(USE_SAMPLE_WEIGHT),
    "sample_weight_min": float(min(w_train.min(), w_val.min(), w_test.min())),
    "sample_weight_max": float(max(w_train.max(), w_val.max(), w_test.max())),
}

Path("reports/preprocessing_summary.json").write_text(
    json.dumps(preprocessing_summary, indent=2),
    encoding="utf-8",
)

print(json.dumps(preprocessing_summary, indent=2))
print("Saved preprocessing summary to: reports/preprocessing_summary.json")


## Kết luận Step 3

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Tổng hợp kết quả preprocessing.

Sau Step 3, Colab runtime cần có sẵn:

- `x_train`, `y_train`, `w_train`
- `x_val`, `y_val`, `w_val`
- `x_test`, `y_test`, `w_test`

Và thư mục `reports/` cần có thêm:

- `preprocessing_config.json`
- `preprocessing_summary.json`
- `preprocessed_samples.png`

Các bước sau mới xử lý augmentation, tạo `tf.data.Dataset`, xây model và train CNN.


## Step 4 - Xử lý mất cân bằng lớp và augmentation

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Phân tích mất cân bằng lớp và tạo augmentation preview.  
> **Phối hợp:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Dùng kết quả `sample_weight` ở bước huấn luyện sau.

Mục tiêu của Step 4:

- Phân tích lại mức độ mất cân bằng lớp sau Step 3.
- Tính `class_weight` để đưa vào báo cáo và làm phương án dự phòng.
- Thống kê `sample_weight` có sẵn trong dataset.
- Chốt chiến lược: **ưu tiên dùng `sample_weight` khi train**, không dùng đồng thời với `class_weight`.
- Tạo augmentation pipeline và ảnh preview.

Step này **chưa tạo `tf.data.Dataset`**, **chưa build model**, và **chưa train model**.


### 4.1. Nhắc lại vấn đề mất cân bằng lớp

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Diễn giải vì sao cần weight khi training.

Từ Step 2, train set bị mất cân bằng rõ rệt: class `happy` nhiều nhất, còn `disgust` ít nhất. Nếu train trực tiếp, model có thể thiên về các class nhiều mẫu và cho F1-score thấp ở class ít mẫu.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Tóm tắt lại mất cân bằng lớp từ train set.
train_counts = class_counts["train"]
majority_class = max(train_counts, key=train_counts.get)
minority_class = min(train_counts, key=train_counts.get)
imbalance_ratio = train_counts[majority_class] / max(train_counts[minority_class], 1)

step4_imbalance_note = {
    "majority_class": majority_class,
    "majority_count": train_counts[majority_class],
    "minority_class": minority_class,
    "minority_count": train_counts[minority_class],
    "imbalance_ratio": round(float(imbalance_ratio), 2),
    "training_risk": "Model may bias toward majority classes if weights are not used.",
}

print(json.dumps(step4_imbalance_note, indent=2))


### 4.2. Tính class weight để phân tích và dự phòng

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Tính trọng số theo class từ phân bố train set.

`class_weight` không phải chiến lược chính của nhóm, nhưng vẫn nên tính để:

- Đưa vào báo cáo nhằm chứng minh mức độ mất cân bằng.
- Có phương án dự phòng nếu không dùng `sample_weight`.
- So sánh với `sample_weight` có sẵn trong dataset.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Tính class_weight để phân tích và lưu báo cáo.
from sklearn.utils.class_weight import compute_class_weight

classes = np.array(sorted(label_mapping.keys()))
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)

class_weight = {
    int(label_id): float(weight)
    for label_id, weight in zip(classes, class_weights_array)
}

class_weight_named = {
    label_mapping[int(label_id)]: float(weight)
    for label_id, weight in zip(classes, class_weights_array)
}

Path("reports/class_weight.json").write_text(
    json.dumps(class_weight_named, indent=2),
    encoding="utf-8",
)

print(json.dumps(class_weight_named, indent=2))
print("Saved class weights to: reports/class_weight.json")


### 4.3. Thống kê sample weight có sẵn

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Phân tích `sample_weight` của dataset.  
> **Phối hợp:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Sử dụng `sample_weight` trong bước huấn luyện sau.

Dataset FER2013-enhanced đã có cột `sample_weight`. Step 4 thống kê min/mean/median/max của weight ở từng split để xác nhận weight tồn tại và có khoảng giá trị hợp lý.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Tóm tắt sample_weight theo từng split.
def summarize_weights(weights):
    return {
        "min": float(np.min(weights)),
        "mean": float(np.mean(weights)),
        "median": float(np.median(weights)),
        "max": float(np.max(weights)),
    }

sample_weight_summary = {
    "uses_sample_weight": bool(USE_SAMPLE_WEIGHT),
    "strategy": "Use sample_weight as the primary imbalance handling method during training.",
    "train": summarize_weights(w_train),
    "validation": summarize_weights(w_val),
    "test": summarize_weights(w_test),
}

Path("reports/sample_weight_summary.json").write_text(
    json.dumps(sample_weight_summary, indent=2),
    encoding="utf-8",
)

print(json.dumps(sample_weight_summary, indent=2))
print("Saved sample weight summary to: reports/sample_weight_summary.json")


### 4.4. Chọn chiến lược weight cho training

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Ghi rõ quyết định xử lý mất cân bằng.  
> **Phối hợp:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Áp dụng quyết định này trong `model.fit` sau.

Nhóm chọn chiến lược:

- **Dùng `sample_weight` là chính khi training.**
- `class_weight` chỉ dùng để phân tích và dự phòng.
- Không dùng đồng thời `sample_weight` và `class_weight`, vì có thể làm weight của class ít mẫu bị khuếch đại quá mức.

Ở bước training sau, data pipeline nên trả về tuple dạng:

```python
(image_batch, label_batch, sample_weight_batch)
```


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Lưu quyết định xử lý mất cân bằng cho báo cáo và bước train sau.
imbalance_strategy = {
    "primary_training_weight": "sample_weight",
    "fallback_training_weight": "class_weight",
    "use_sample_weight_and_class_weight_together": False,
    "reason": (
        "FER2013-enhanced already provides sample_weight. class_weight is computed for "
        "analysis and fallback only, to avoid over-amplifying minority classes."
    ),
}

Path("reports/imbalance_strategy.json").write_text(
    json.dumps(imbalance_strategy, indent=2),
    encoding="utf-8",
)

print(json.dumps(imbalance_strategy, indent=2))
print("Saved imbalance strategy to: reports/imbalance_strategy.json")


### 4.5. Định nghĩa augmentation pipeline

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Tạo augmentation để mô phỏng điều kiện webcam thực tế.

Augmentation chỉ áp dụng cho train set ở bước training sau. Validation/test giữ nguyên để đánh giá công bằng.

Các phép augmentation được chọn nhẹ vì ảnh chỉ `48x48`:

- Flip ngang.
- Xoay nhẹ.
- Dịch chuyển nhẹ.

Vòng đầu **không dùng zoom** vì ảnh FER chỉ `48x48`; zoom dễ làm ảnh mờ và mất chi tiết mắt/miệng.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Định nghĩa augmentation pipeline cho train set.
AUG_RANDOM_ROTATION = 0.02
AUG_RANDOM_TRANSLATION = 0.02

# Set seed để augmentation preview có thể tái lập tương đối giữa các lần chạy.
tf.keras.utils.set_random_seed(42)

data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(AUG_RANDOM_ROTATION),
        tf.keras.layers.RandomTranslation(AUG_RANDOM_TRANSLATION, AUG_RANDOM_TRANSLATION),
    ],
    name="data_augmentation",
)

augmentation_config = {
    "random_flip": "horizontal",
    "random_rotation": AUG_RANDOM_ROTATION,
    "random_zoom": None,
    "random_translation_height": AUG_RANDOM_TRANSLATION,
    "random_translation_width": AUG_RANDOM_TRANSLATION,
    "applied_to": "train only",
    "not_applied_to": ["validation", "test"],
    "reason": "Use very light augmentation because FER images are small 48x48 grayscale faces and stronger transforms can blur facial details.",
}

Path("reports/augmentation_config.json").write_text(
    json.dumps(augmentation_config, indent=2),
    encoding="utf-8",
)

print(json.dumps(augmentation_config, indent=2))
print("Saved augmentation config to: reports/augmentation_config.json")


### 4.6. Xem ảnh augmentation preview

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Kiểm tra augmentation không phá hỏng ảnh.

Cell này tạo ảnh preview gồm hai hàng:

- Hàng 1: ảnh gốc sau preprocessing.
- Hàng 2: ảnh sau augmentation.

Nếu ảnh augmented bị méo quá mạnh hoặc mất khuôn mặt, cần giảm thông số augmentation.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Tạo và lưu ảnh minh họa augmentation.
num_preview = min(8, len(x_train))
original_images = x_train[:num_preview]
augmented_images = data_augmentation(original_images, training=True).numpy()
augmented_images = np.clip(augmented_images, 0.0, 1.0).astype("float32")

fig, axes = plt.subplots(2, num_preview, figsize=(2.2 * num_preview, 5))
if num_preview == 1:
    axes = np.array([[axes[0]], [axes[1]]])

for index in range(num_preview):
    axes[0, index].imshow(original_images[index].squeeze(), cmap="gray", vmin=0.0, vmax=1.0)
    axes[0, index].set_title(label_mapping[int(y_train[index])])
    axes[0, index].axis("off")

    axes[1, index].imshow(augmented_images[index].squeeze(), cmap="gray", vmin=0.0, vmax=1.0)
    axes[1, index].axis("off")

axes[0, 0].set_ylabel("Original")
axes[1, 0].set_ylabel("Augmented")
plt.tight_layout()
plt.savefig("reports/augmentation_examples.png", dpi=160, bbox_inches="tight")
plt.show()
print("Saved augmentation preview to: reports/augmentation_examples.png")


### 4.7. Validate augmentation output

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Kiểm tra shape, dtype và pixel range sau augmentation.

Cell này đảm bảo augmentation không làm sai shape đầu vào của CNN và pixel vẫn nằm trong khoảng `[0,1]`.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Validate augmented images trước khi dùng ở training step sau.
if augmented_images.shape != original_images.shape:
    raise ValueError(f"Augmented shape mismatch: {augmented_images.shape} != {original_images.shape}")
if augmented_images.dtype != np.float32:
    raise TypeError(f"Augmented dtype mismatch: {augmented_images.dtype}")
if float(augmented_images.min()) < 0.0 or float(augmented_images.max()) > 1.0:
    raise ValueError("Augmented pixel range is outside [0, 1]")

augmentation_summary = {
    "preview_input_shape": list(original_images.shape),
    "preview_output_shape": list(augmented_images.shape),
    "augmented_dtype": str(augmented_images.dtype),
    "augmented_pixel_min": float(augmented_images.min()),
    "augmented_pixel_max": float(augmented_images.max()),
    "validation_passed": True,
}

Path("reports/augmentation_summary.json").write_text(
    json.dumps(augmentation_summary, indent=2),
    encoding="utf-8",
)

print(json.dumps(augmentation_summary, indent=2))
print("Saved augmentation summary to: reports/augmentation_summary.json")


## Kết luận Step 4

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Tổng hợp kết quả xử lý mất cân bằng và augmentation.  
> **Phối hợp:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Dùng các kết quả này ở bước huấn luyện sau.

Sau Step 4, Colab runtime cần có:

- `class_weight`
- `class_weight_named`
- `sample_weight_summary`
- `imbalance_strategy`
- `data_augmentation`

Và thư mục `reports/` cần có thêm:

- `class_weight.json`
- `sample_weight_summary.json`
- `imbalance_strategy.json`
- `augmentation_config.json`
- `augmentation_examples.png`
- `augmentation_summary.json`

Quyết định chính: bước training sau sẽ ưu tiên dùng `sample_weight`; `class_weight` chỉ là phân tích/dự phòng.


## Step 5 - Tạo `tf.data` pipeline cho training

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Đóng gói dữ liệu đã preprocess, sample weight và augmentation thành pipeline huấn luyện.  
> **Phối hợp:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Dùng trực tiếp `train_ds`, `val_ds`, `test_ds` khi xây và train CNN.

Mục tiêu của Step 5 là chuẩn bị dữ liệu đúng format cho `model.fit`:

```python
model.fit(train_ds, validation_data=val_ds, ...)
```

Quy tắc quan trọng:

- Train set có shuffle và augmentation.
- Validation/test không augmentation để đánh giá công bằng.
- Dùng `sample_weight` là weight chính khi training.
- Dataset trả về tuple dạng `(image_batch, label_batch, sample_weight_batch)`.


### 5.1. Cấu hình batch và hiệu năng pipeline

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Chọn cấu hình đơn giản, đủ ổn định cho Colab T4.

`BATCH_SIZE = 64` là lựa chọn cân bằng cho ảnh `48x48` grayscale: đủ nhanh, ít rủi ro hết RAM/GPU memory.

Dataset không bật `.cache()` mặc định vì toàn bộ ảnh đã nằm trong mảng NumPy `x_train`, `x_val`, `x_test`; cache thêm có thể làm tốn RAM không cần thiết trên Colab. `prefetch` vẫn được bật để pipeline mượt hơn khi training.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Cấu hình tf.data pipeline cho bước training sau.
BATCH_SIZE = 64
SHUFFLE_BUFFER_SIZE = min(len(x_train), 5000)
USE_DATASET_CACHE = False
AUTOTUNE = tf.data.AUTOTUNE

tfdata_config = {
    "batch_size": BATCH_SIZE,
    "shuffle_buffer_size": SHUFFLE_BUFFER_SIZE,
    "cache_enabled": USE_DATASET_CACHE,
    "prefetch_enabled": True,
    "train_augmentation_enabled": True,
    "validation_augmentation_enabled": False,
    "test_augmentation_enabled": False,
    "uses_sample_weight": bool(USE_SAMPLE_WEIGHT),
}

print(json.dumps(tfdata_config, indent=2))


### 5.2. Hàm tạo dataset cho từng split

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Viết hàm dùng chung cho train/validation/test.

Hàm này giữ logic thật rõ:

- Train: shuffle -> batch -> augmentation -> prefetch.
- Validation/test: batch -> prefetch.
- Cả ba split đều giữ `sample_weight` trong output.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Tạo helper build tf.data.Dataset có sample_weight.
def build_weighted_dataset(images, labels, weights, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((images, labels, weights))

    if training:
        dataset = dataset.shuffle(
            buffer_size=SHUFFLE_BUFFER_SIZE,
            seed=42,
            reshuffle_each_iteration=True,
        )

    if USE_DATASET_CACHE:
        dataset = dataset.cache()

    dataset = dataset.batch(BATCH_SIZE)

    if training:
        dataset = dataset.map(
            lambda image_batch, label_batch, weight_batch: (
                data_augmentation(image_batch, training=True),
                label_batch,
                weight_batch,
            ),
            num_parallel_calls=AUTOTUNE,
        )

    dataset = dataset.prefetch(AUTOTUNE)
    return dataset


### 5.3. Khởi tạo `train_ds`, `val_ds`, `test_ds`

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Tạo ba dataset chính để Nguyễn Minh Quân (20235816, trưởng nhóm) dùng khi train CNN.

Sau cell này, các bước model chỉ cần dùng:

```python
model.fit(train_ds, validation_data=val_ds, ...)
model.evaluate(test_ds)
```


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Khởi tạo dataset cho train/validation/test.
train_ds = build_weighted_dataset(x_train, y_train, w_train, training=True)
val_ds = build_weighted_dataset(x_val, y_val, w_val, training=False)
test_ds = build_weighted_dataset(x_test, y_test, w_test, training=False)

train_batches = int(np.ceil(len(x_train) / BATCH_SIZE))
val_batches = int(np.ceil(len(x_val) / BATCH_SIZE))
test_batches = int(np.ceil(len(x_test) / BATCH_SIZE))

print(f"train_ds batches: {train_batches}")
print(f"val_ds batches: {val_batches}")
print(f"test_ds batches: {test_batches}")


### 5.4. Validate batch đầu ra

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Kiểm tra dataset output đúng format cho `model.fit`.

Cell này kiểm tra shape, dtype, label range, pixel range và sample weight trước khi chuyển sang phần model.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Validate batch đầu tiên của train/validation/test dataset.
def validate_dataset_batch(dataset, split_name):
    image_batch, label_batch, weight_batch = next(iter(dataset))

    if image_batch.shape[1:] != INPUT_SHAPE:
        raise ValueError(f"{split_name}: image shape mismatch: {image_batch.shape}")
    if len(label_batch.shape) != 1:
        raise ValueError(f"{split_name}: labels must be rank 1, got {label_batch.shape}")
    if len(weight_batch.shape) != 1:
        raise ValueError(f"{split_name}: sample weights must be rank 1, got {weight_batch.shape}")
    if image_batch.shape[0] != label_batch.shape[0] or image_batch.shape[0] != weight_batch.shape[0]:
        raise ValueError(f"{split_name}: batch size mismatch")

    pixel_min = float(tf.reduce_min(image_batch).numpy())
    pixel_max = float(tf.reduce_max(image_batch).numpy())
    label_min = int(tf.reduce_min(label_batch).numpy())
    label_max = int(tf.reduce_max(label_batch).numpy())
    weight_min = float(tf.reduce_min(weight_batch).numpy())
    weight_max = float(tf.reduce_max(weight_batch).numpy())

    if pixel_min < 0.0 or pixel_max > 1.0:
        raise ValueError(f"{split_name}: pixel range outside [0, 1]: {pixel_min}..{pixel_max}")
    if label_min < 0 or label_max >= NUM_CLASSES:
        raise ValueError(f"{split_name}: label range invalid: {label_min}..{label_max}")
    if weight_min <= 0.0:
        raise ValueError(f"{split_name}: sample_weight must be positive, min={weight_min}")

    return {
        "image_shape": list(image_batch.shape),
        "image_dtype": image_batch.dtype.name,
        "label_shape": list(label_batch.shape),
        "label_dtype": label_batch.dtype.name,
        "weight_shape": list(weight_batch.shape),
        "weight_dtype": weight_batch.dtype.name,
        "pixel_min": pixel_min,
        "pixel_max": pixel_max,
        "label_min": label_min,
        "label_max": label_max,
        "sample_weight_min": weight_min,
        "sample_weight_max": weight_max,
    }

batch_validation_summary = {
    "train": validate_dataset_batch(train_ds, "train"),
    "validation": validate_dataset_batch(val_ds, "validation"),
    "test": validate_dataset_batch(test_ds, "test"),
}

print(json.dumps(batch_validation_summary, indent=2))


### 5.5. Preview batch train sau augmentation

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Lưu bằng chứng augmentation đã nằm trong train pipeline.

Khác với Step 4 chỉ preview trực tiếp từ `data_augmentation`, cell này lấy ảnh từ chính `train_ds`. Vì vậy ảnh preview phản ánh đúng dữ liệu mà model sẽ thấy khi training.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Lưu preview một batch từ train_ds sau augmentation.
preview_image_batch, preview_label_batch, preview_weight_batch = next(iter(train_ds))
num_preview = min(12, int(preview_image_batch.shape[0]))

fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
axes = axes.flatten()

for index in range(len(axes)):
    axes[index].axis("off")
    if index < num_preview:
        label_id = int(preview_label_batch[index].numpy())
        weight_value = float(preview_weight_batch[index].numpy())
        axes[index].imshow(
            preview_image_batch[index].numpy().squeeze(),
            cmap="gray",
            vmin=0.0,
            vmax=1.0,
        )
        axes[index].set_title(f"{label_mapping[label_id]} | w={weight_value:.2f}", fontsize=9)

plt.tight_layout()
plt.savefig("reports/tfdata_augmented_batch_preview.png", dpi=160, bbox_inches="tight")
plt.show()
print("Saved tf.data preview to: reports/tfdata_augmented_batch_preview.png")


### 5.6. Lưu summary của `tf.data` pipeline

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Lưu cấu hình và kết quả validate để đưa vào báo cáo.

File summary này giúp nhóm chứng minh pipeline train/validation/test đã được chuẩn hóa trước khi Nguyễn Minh Quân (20235816, trưởng nhóm) xây CNN.


In [ ]:
# Phụ trách: Đặng Hoàng Quân (20235813) - Lưu summary Step 5.
tfdata_pipeline_summary = {
    **tfdata_config,
    "train_samples": int(len(x_train)),
    "validation_samples": int(len(x_val)),
    "test_samples": int(len(x_test)),
    "train_batches": train_batches,
    "validation_batches": val_batches,
    "test_batches": test_batches,
    "output_tuple": ["image_batch", "label_batch", "sample_weight_batch"],
    "batch_validation": batch_validation_summary,
    "reports": [
        "reports/tfdata_augmented_batch_preview.png",
        "reports/tfdata_pipeline_summary.json",
    ],
}

Path("reports/tfdata_pipeline_summary.json").write_text(
    json.dumps(tfdata_pipeline_summary, indent=2),
    encoding="utf-8",
)

print(json.dumps(tfdata_pipeline_summary, indent=2))
print("Saved tf.data pipeline summary to: reports/tfdata_pipeline_summary.json")


## Kết luận Step 5

> **Phụ trách:** Đặng Hoàng Quân (20235813) - Tổng hợp pipeline dữ liệu đã sẵn sàng cho CNN.  
> **Phối hợp:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Dùng các biến `train_ds`, `val_ds`, `test_ds` ở bước xây và train model.

Sau Step 5, Colab runtime cần có:

- `train_ds`
- `val_ds`
- `test_ds`
- `tfdata_pipeline_summary`

Và thư mục `reports/` cần có thêm:

- `tfdata_augmented_batch_preview.png`
- `tfdata_pipeline_summary.json`

Khi sang phần model, nhóm dùng `sample_weight` thông qua dataset tuple `(image_batch, label_batch, sample_weight_batch)`, không cần truyền thêm `class_weight` vào `model.fit`.


## Step 6 - Xây kiến trúc Custom CNN chính

> **Phụ trách:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Thiết kế CNN chính của nhóm.  
> **Phối hợp:** Đặng Hoàng Quân (20235813) - Cung cấp `INPUT_SHAPE`, `NUM_CLASSES`, `train_ds`, `val_ds`, `test_ds` từ các bước trước.

Mục tiêu của Step 6 là xây và compile một model CNN tự thiết kế, không dùng DeepFace hoặc pretrained model. Step này **chưa train model**; training sẽ thực hiện ở Step 7.

Model nhận ảnh grayscale kích thước `(48, 48, 1)` và dự đoán 7 lớp cảm xúc.


### 6.1. Định hướng kiến trúc

> **Phụ trách:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Giải thích lựa chọn kiến trúc CNN.

Nhóm chọn một CNN chính gồm 4 block convolution:

- Block 1: 32 filters.
- Block 2: 64 filters.
- Block 3: 128 filters.
- Block 4: 256 filters.

Mỗi block dùng `Conv2D -> BatchNormalization -> ReLU`, sau đó `MaxPooling2D` và `Dropout`. Cách này phù hợp với ảnh FER `48x48` vì sau 4 lần pooling, feature map còn khoảng `3x3`, vẫn đủ nhỏ để học đặc trưng cấp cao mà không quá nặng cho Colab T4.

Phần head dùng `GlobalAveragePooling2D` thay vì `Flatten` để giảm số tham số và giảm overfit.


### 6.2. Cấu hình model

> **Phụ trách:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Khai báo tên model, learning rate và seed.

Các giá trị này được lưu lại để báo cáo và để Step 7 dùng khi training.


In [ ]:
# Phụ trách: Nguyễn Minh Quân (20235816, trưởng nhóm) - Cấu hình model CNN chính.
MODEL_NAME = "custom_cnn_v1"
MODEL_SEED = 42
LEARNING_RATE = 1e-3

tf.keras.utils.set_random_seed(MODEL_SEED)

model_config = {
    "model_name": MODEL_NAME,
    "model_type": "custom_cnn",
    "input_shape": list(INPUT_SHAPE),
    "num_classes": int(NUM_CLASSES),
    "conv_filters": [32, 64, 128, 256],
    "dropout_rates": [0.15, 0.20, 0.25, 0.30, 0.40],
    "dense_units": 256,
    "learning_rate": LEARNING_RATE,
    "optimizer": "Adam",
    "loss": "sparse_categorical_crossentropy",
    "uses_pretrained_model": False,
    "uses_deepface": False,
    "training_done_in_this_step": False,
}

print(json.dumps(model_config, indent=2))


### 6.3. Tạo helper cho convolution block

> **Phụ trách:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Viết block CNN ngắn gọn, tái sử dụng được trong model chính.

Mỗi block có hai lớp convolution để học đặc trưng tốt hơn trước khi giảm kích thước bằng pooling.


In [ ]:
# Phụ trách: Nguyễn Minh Quân (20235816, trưởng nhóm) - Định nghĩa block Conv-BatchNorm-ReLU-Pool-Dropout.
def conv_block(inputs, filters, dropout_rate, block_name):
    x = tf.keras.layers.Conv2D(
        filters,
        kernel_size=(3, 3),
        padding="same",
        use_bias=False,
        name=f"{block_name}_conv1",
    )(inputs)
    x = tf.keras.layers.BatchNormalization(name=f"{block_name}_bn1")(x)
    x = tf.keras.layers.Activation("relu", name=f"{block_name}_relu1")(x)

    x = tf.keras.layers.Conv2D(
        filters,
        kernel_size=(3, 3),
        padding="same",
        use_bias=False,
        name=f"{block_name}_conv2",
    )(x)
    x = tf.keras.layers.BatchNormalization(name=f"{block_name}_bn2")(x)
    x = tf.keras.layers.Activation("relu", name=f"{block_name}_relu2")(x)

    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2), name=f"{block_name}_pool")(x)
    x = tf.keras.layers.Dropout(dropout_rate, name=f"{block_name}_dropout")(x)
    return x


### 6.4. Xây model `custom_cnn_v1`

> **Phụ trách:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Xây kiến trúc CNN 4 block.

Model này là model chính đầu tiên của nhóm. Nếu cần thử biến thể, nhóm sẽ làm sau khi đã có kết quả training chuẩn từ Step 7.


In [ ]:
# Phụ trách: Nguyễn Minh Quân (20235816, trưởng nhóm) - Xây model CNN chính custom_cnn_v1.
def build_custom_cnn_v1(input_shape, num_classes):
    inputs = tf.keras.Input(shape=input_shape, name="input_image")

    x = conv_block(inputs, filters=32, dropout_rate=0.15, block_name="block1")
    x = conv_block(x, filters=64, dropout_rate=0.20, block_name="block2")
    x = conv_block(x, filters=128, dropout_rate=0.25, block_name="block3")
    x = conv_block(x, filters=256, dropout_rate=0.30, block_name="block4")

    x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = tf.keras.layers.Dense(256, use_bias=False, name="classifier_dense")(x)
    x = tf.keras.layers.BatchNormalization(name="classifier_bn")(x)
    x = tf.keras.layers.Activation("relu", name="classifier_relu")(x)
    x = tf.keras.layers.Dropout(0.40, name="classifier_dropout")(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="emotion_probabilities")(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs, name=MODEL_NAME)

model = build_custom_cnn_v1(INPUT_SHAPE, NUM_CLASSES)
model.summary()


### 6.5. Compile model

> **Phụ trách:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Compile model để sẵn sàng training ở Step 7.

Loss dùng `sparse_categorical_crossentropy` vì label đang là số nguyên `0..6`, không phải one-hot.


In [ ]:
# Phụ trách: Nguyễn Minh Quân (20235816, trưởng nhóm) - Compile model CNN chính.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name="top_2_accuracy"),
    ],
)

print(f"Compiled {MODEL_NAME} with Adam learning_rate={LEARNING_RATE}")


### 6.6. Validate kiến trúc model

> **Phụ trách:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Kiểm tra input/output và softmax trước khi train.

Cell này chưa train model. Nó chỉ đảm bảo model nhận đúng shape và output là phân phối xác suất cho 7 lớp.


In [ ]:
# Phụ trách: Nguyễn Minh Quân (20235816, trưởng nhóm) - Validate model bằng dummy input.
dummy_input = tf.zeros((2, *INPUT_SHAPE), dtype=tf.float32)
dummy_output = model(dummy_input, training=False)
softmax_sums = tf.reduce_sum(dummy_output, axis=1).numpy()

if model.input_shape != (None, *INPUT_SHAPE):
    raise ValueError(f"Input shape mismatch: {model.input_shape}")
if model.output_shape != (None, NUM_CLASSES):
    raise ValueError(f"Output shape mismatch: {model.output_shape}")
if dummy_output.shape != (2, NUM_CLASSES):
    raise ValueError(f"Dummy output shape mismatch: {dummy_output.shape}")
if not np.allclose(softmax_sums, np.ones_like(softmax_sums), atol=1e-5):
    raise ValueError(f"Softmax rows do not sum to 1: {softmax_sums}")

model_validation_summary = {
    "input_shape": list(model.input_shape),
    "output_shape": list(model.output_shape),
    "dummy_output_shape": list(dummy_output.shape),
    "softmax_row_sums": softmax_sums.tolist(),
    "validation_passed": True,
}

print(json.dumps(model_validation_summary, indent=2))


### 6.7. Lưu summary kiến trúc model

> **Phụ trách:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Lưu thông tin kiến trúc để đưa vào báo cáo.

Step 6 chỉ lưu summary kiến trúc. Model `.keras` sẽ được lưu sau khi training ở Step 7 để tránh nhầm model chưa train với model đã train.


In [ ]:
# Phụ trách: Nguyễn Minh Quân (20235816, trưởng nhóm) - Lưu summary kiến trúc model.
model_architecture_summary = {
    **model_config,
    "keras_model_name": model.name,
    "total_params": int(model.count_params()),
    "input_shape_runtime": list(model.input_shape),
    "output_shape_runtime": list(model.output_shape),
    "validation": model_validation_summary,
    "notes": [
        "Four convolution blocks are used for stronger feature extraction than a simple baseline CNN.",
        "GlobalAveragePooling2D is used instead of Flatten to reduce overfitting risk.",
        "The model is compiled but not trained in Step 6.",
    ],
}

Path("reports/model_architecture_summary.json").write_text(
    json.dumps(model_architecture_summary, indent=2),
    encoding="utf-8",
)

print(json.dumps(model_architecture_summary, indent=2))
print("Saved model architecture summary to: reports/model_architecture_summary.json")


## Kết luận Step 6

> **Phụ trách:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Tổng hợp model CNN chính.  
> **Phối hợp:** Đinh Văn Phạm Việt (20235870) - Dùng kết quả model đã train để đánh giá ở Step 8.

Sau Step 6, Colab runtime cần có:

- `model`
- `model_config`
- `model_validation_summary`
- `model_architecture_summary`

Và thư mục `reports/` cần có thêm:

- `model_architecture_summary.json`

Step này chưa train và chưa lưu file `.keras`. Model sẽ được Nguyễn Minh Quân (20235816, trưởng nhóm) train và lưu checkpoint ở Step 7; Đinh Văn Phạm Việt (20235870) sẽ đánh giá kỹ hơn ở Step 8.


## Step 7 - Train `custom_cnn_v1`

> **Phụ trách:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Train model CNN chính và lưu checkpoint tốt nhất.  
> **Phối hợp:** Đinh Văn Phạm Việt (20235870) - Nhận kết quả training để đánh giá sâu ở Step 8.

Mục tiêu của Step 7 là train model bằng `train_ds`, validate bằng `val_ds`, lưu best checkpoint và lưu đầy đủ lịch sử training.

Step này chưa đánh giá trên test set. Test set sẽ để Step 8 nhằm giữ đánh giá cuối cùng khách quan hơn.


### 7.1. Cấu hình training

> **Phụ trách:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Chọn training settings phù hợp với CNN 4 block.

Nhóm dùng tối đa `50` epochs nhưng có `EarlyStopping`, nên training có thể dừng sớm nếu validation loss không cải thiện.

Optimizer chính là `AdamW` với `weight_decay=1e-4` để regularize model tốt hơn Adam thường. Nếu môi trường TensorFlow không hỗ trợ `AdamW`, notebook sẽ fallback sang `Adam`.


In [ ]:
# Phụ trách: Nguyễn Minh Quân (20235816, trưởng nhóm) - Cấu hình training cho custom_cnn_v1.
EPOCHS = 50
EARLY_STOPPING_PATIENCE = 8
REDUCE_LR_PATIENCE = 3
REDUCE_LR_FACTOR = 0.5
MIN_LEARNING_RATE = 1e-6
WEIGHT_DECAY = 1e-4

BEST_MODEL_PATH = Path("models/custom_cnn_v1_best.keras")
TRAINING_LOG_PATH = Path("reports/custom_cnn_v1_training_log.csv")
HISTORY_PATH = Path("reports/custom_cnn_v1_history.json")
TRAINING_CURVES_PATH = Path("reports/custom_cnn_v1_training_curves.png")
VALIDATION_RESULTS_PATH = Path("reports/custom_cnn_v1_validation_results.json")
TRAINING_SUMMARY_PATH = Path("reports/custom_cnn_v1_training_summary.json")

training_config = {
    "model_name": MODEL_NAME,
    "epochs_requested": EPOCHS,
    "batch_size": BATCH_SIZE,
    "optimizer_preferred": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "loss": "sparse_categorical_crossentropy",
    "monitor_metric": "val_loss",
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "reduce_lr_patience": REDUCE_LR_PATIENCE,
    "reduce_lr_factor": REDUCE_LR_FACTOR,
    "min_learning_rate": MIN_LEARNING_RATE,
    "uses_sample_weight_from_dataset": True,
    "uses_class_weight": False,
    "evaluates_test_set_in_this_step": False,
}

print(json.dumps(training_config, indent=2))


### 7.2. Re-compile model bằng AdamW

> **Phụ trách:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Compile lại model bằng optimizer chính thức cho training.

Step 6 compile model để validate kiến trúc. Step 7 compile lại bằng `AdamW` vì đây là cấu hình training chính thức của nhóm.

Loss giữ `sparse_categorical_crossentropy` vì label là số nguyên `0..6` và output là softmax 7 lớp.


In [ ]:
# Phụ trách: Nguyễn Minh Quân (20235816, trưởng nhóm) - Re-compile model với AdamW nếu TensorFlow hỗ trợ.
try:
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    optimizer_name = "AdamW"
except AttributeError:
    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    optimizer_name = "Adam"
    training_config["optimizer_fallback_reason"] = "tf.keras.optimizers.AdamW is not available in this TensorFlow version."

training_config["optimizer_used"] = optimizer_name

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name="top_2_accuracy"),
    ],
)

print(f"Compiled {MODEL_NAME} with {optimizer_name}, learning_rate={LEARNING_RATE}")


### 7.3. Tạo callbacks

> **Phụ trách:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Tạo checkpoint, early stopping, reduce LR và CSV log.

Callbacks giúp training ổn định hơn:

- `ModelCheckpoint`: lưu model có validation loss tốt nhất.
- `EarlyStopping`: dừng khi validation loss không cải thiện.
- `ReduceLROnPlateau`: giảm learning rate khi model bị chững.
- `CSVLogger`: lưu log từng epoch để báo cáo.


In [ ]:
# Phụ trách: Nguyễn Minh Quân (20235816, trưởng nhóm) - Tạo callbacks cho training.
BEST_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
TRAINING_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=REDUCE_LR_FACTOR,
        patience=REDUCE_LR_PATIENCE,
        min_lr=MIN_LEARNING_RATE,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(
        filename=str(TRAINING_LOG_PATH),
        separator=",",
        append=False,
    ),
]

print("Callbacks ready:")
for callback in callbacks:
    print(f"- {callback.__class__.__name__}")


### 7.4. Train model

> **Phụ trách:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Train `custom_cnn_v1` bằng `train_ds` và validate bằng `val_ds`.

Không truyền `class_weight` vào `model.fit`, vì `train_ds` đã trả về `sample_weight_batch` trong tuple `(image_batch, label_batch, sample_weight_batch)`.


In [ ]:
# Phụ trách: Nguyễn Minh Quân (20235816, trưởng nhóm) - Train model CNN chính.
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

epochs_trained = len(history.history.get("loss", []))
print(f"Training finished after {epochs_trained} epochs.")
print(f"Best model checkpoint path: {BEST_MODEL_PATH}")


### 7.5. Lưu training history

> **Phụ trách:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Lưu lịch sử train/validation theo từng epoch.

File history giúp nhóm vẽ biểu đồ, phân tích overfit/underfit và đưa số liệu vào báo cáo.


In [ ]:
# Phụ trách: Nguyễn Minh Quân (20235816, trưởng nhóm) - Lưu history object ra JSON.
def history_to_jsonable(history_dict):
    jsonable = {}
    for key, values in history_dict.items():
        jsonable[key] = [float(value) for value in values]
    return jsonable

history_dict = history_to_jsonable(history.history)

HISTORY_PATH.write_text(
    json.dumps(history_dict, indent=2),
    encoding="utf-8",
)

print(json.dumps({key: values[-3:] for key, values in history_dict.items()}, indent=2))
print(f"Saved training history to: {HISTORY_PATH}")


### 7.6. Vẽ training curves

> **Phụ trách:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Trực quan hóa loss, accuracy và top-2 accuracy.

Biểu đồ này dùng để kiểm tra model đang học ổn, overfit hay underfit.


In [ ]:
# Phụ trách: Nguyễn Minh Quân (20235816, trưởng nhóm) - Vẽ và lưu training curves.
def plot_metric(ax, history_dict, train_key, val_key, title, ylabel):
    epochs = range(1, len(history_dict[train_key]) + 1)
    ax.plot(epochs, history_dict[train_key], label=f"train {train_key}")
    if val_key in history_dict:
        ax.plot(epochs, history_dict[val_key], label=f"validation {val_key}")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_metric(axes[0], history_dict, "loss", "val_loss", "Loss", "Loss")
plot_metric(axes[1], history_dict, "accuracy", "val_accuracy", "Accuracy", "Accuracy")
plot_metric(axes[2], history_dict, "top_2_accuracy", "val_top_2_accuracy", "Top-2 Accuracy", "Top-2 Accuracy")

plt.tight_layout()
plt.savefig(TRAINING_CURVES_PATH, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved training curves to: {TRAINING_CURVES_PATH}")


### 7.7. Load best checkpoint và evaluate validation

> **Phụ trách:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Kiểm tra checkpoint tốt nhất có thể load và evaluate được.

Đây chỉ là đánh giá nhanh trên validation set. Test set vẫn để Step 8.


In [ ]:
# Phụ trách: Nguyễn Minh Quân (20235816, trưởng nhóm) - Load best model và evaluate trên validation set.
if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"Best model checkpoint not found: {BEST_MODEL_PATH}")

best_model = tf.keras.models.load_model(BEST_MODEL_PATH)
validation_results_raw = best_model.evaluate(val_ds, verbose=1, return_dict=True)
validation_results = {
    metric_name: float(metric_value)
    for metric_name, metric_value in validation_results_raw.items()
}

VALIDATION_RESULTS_PATH.write_text(
    json.dumps(validation_results, indent=2),
    encoding="utf-8",
)

print(json.dumps(validation_results, indent=2))
print(f"Saved validation results to: {VALIDATION_RESULTS_PATH}")


### 7.8. Lưu training summary

> **Phụ trách:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Tổng hợp kết quả training để báo cáo và chuyển cho Step 8.

Summary này ghi lại epoch tốt nhất, metric validation tốt nhất, optimizer đã dùng và các file output sinh ra.


In [ ]:
# Phụ trách: Nguyễn Minh Quân (20235816, trưởng nhóm) - Lưu summary Step 7.
def get_best_epoch_and_value(history_dict, metric_name, mode="min"):
    values = history_dict.get(metric_name, [])
    if not values:
        return None, None
    if mode == "min":
        best_index = int(np.argmin(values))
    else:
        best_index = int(np.argmax(values))
    return best_index + 1, float(values[best_index])

best_val_loss_epoch, best_val_loss = get_best_epoch_and_value(history_dict, "val_loss", mode="min")
best_val_accuracy_epoch, best_val_accuracy = get_best_epoch_and_value(history_dict, "val_accuracy", mode="max")
best_val_top2_epoch, best_val_top2_accuracy = get_best_epoch_and_value(history_dict, "val_top_2_accuracy", mode="max")

training_summary = {
    **training_config,
    "optimizer_used": optimizer_name,
    "epochs_trained": int(epochs_trained),
    "best_val_loss_epoch": best_val_loss_epoch,
    "best_val_loss": best_val_loss,
    "best_val_accuracy_epoch": best_val_accuracy_epoch,
    "best_val_accuracy": best_val_accuracy,
    "best_val_top_2_accuracy_epoch": best_val_top2_epoch,
    "best_val_top_2_accuracy": best_val_top2_accuracy,
    "validation_results_from_best_checkpoint": validation_results,
    "best_model_path": str(BEST_MODEL_PATH),
    "generated_files": [
        str(BEST_MODEL_PATH),
        str(TRAINING_LOG_PATH),
        str(HISTORY_PATH),
        str(TRAINING_CURVES_PATH),
        str(VALIDATION_RESULTS_PATH),
        str(TRAINING_SUMMARY_PATH),
    ],
}

TRAINING_SUMMARY_PATH.write_text(
    json.dumps(training_summary, indent=2),
    encoding="utf-8",
)

print(json.dumps(training_summary, indent=2))
print(f"Saved training summary to: {TRAINING_SUMMARY_PATH}")


## Kết luận Step 7

> **Phụ trách:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Tổng hợp kết quả train model chính.  
> **Phối hợp:** Đinh Văn Phạm Việt (20235870) - Dùng checkpoint và reports từ Step 7 để đánh giá sâu ở Step 8.

Sau Step 7, Colab runtime cần có:

- `history`
- `best_model`
- `training_summary`

Và file system cần có:

- `models/custom_cnn_v1_best.keras`
- `reports/custom_cnn_v1_training_log.csv`
- `reports/custom_cnn_v1_history.json`
- `reports/custom_cnn_v1_training_curves.png`
- `reports/custom_cnn_v1_validation_results.json`
- `reports/custom_cnn_v1_training_summary.json`

Step 7 chỉ evaluate validation set. Test set sẽ được đánh giá kỹ ở Step 8 bằng confusion matrix, classification report và phân tích lỗi theo từng lớp.


## Step 8 - Đánh giá model trên test set

> **Phụ trách:** Đinh Văn Phạm Việt (20235870) - Đánh giá model đã train trên test set và phân tích lỗi.  
> **Phối hợp:** Nguyễn Minh Quân (20235816, trưởng nhóm) - Cung cấp checkpoint `models/custom_cnn_v1_best.keras` từ Step 7.

Mục tiêu của Step 8 là đánh giá khách quan model tốt nhất trên test set. Step này **không train thêm** và **không chỉnh weights**.

Nhóm sẽ báo cáo cả hai nhóm metric:

- **Weighted metrics** từ `test_ds`, vì `test_ds` có `sample_weight`.
- **Unweighted metrics** từ `x_test`, `y_test`, để dễ so sánh và trình bày khách quan hơn.


### 8.1. Load best checkpoint

> **Phụ trách:** Đinh Văn Phạm Việt (20235870) - Load model tốt nhất đã lưu từ Step 7.

Cell này kiểm tra checkpoint tồn tại và model có input/output shape đúng trước khi đánh giá.


In [ ]:
# Phụ trách: Đinh Văn Phạm Việt (20235870) - Load best checkpoint để đánh giá test set.
EVALUATION_MODEL_PATH = Path("models/custom_cnn_v1_best.keras")

if not EVALUATION_MODEL_PATH.exists():
    raise FileNotFoundError(f"Best model checkpoint not found: {EVALUATION_MODEL_PATH}")

evaluation_model = tf.keras.models.load_model(EVALUATION_MODEL_PATH)

if evaluation_model.input_shape != (None, *INPUT_SHAPE):
    raise ValueError(f"Input shape mismatch: {evaluation_model.input_shape}")
if evaluation_model.output_shape != (None, NUM_CLASSES):
    raise ValueError(f"Output shape mismatch: {evaluation_model.output_shape}")

print(f"Loaded model from: {EVALUATION_MODEL_PATH}")
print(f"Input shape: {evaluation_model.input_shape}")
print(f"Output shape: {evaluation_model.output_shape}")


### 8.2. Weighted evaluation bằng `test_ds`

> **Phụ trách:** Đinh Văn Phạm Việt (20235870) - Đánh giá test set có áp dụng `sample_weight`.

Kết quả này dùng cùng format với Step 7 vì dataset trả về `(image, label, sample_weight)`.


In [ ]:
# Phụ trách: Đinh Văn Phạm Việt (20235870) - Evaluate weighted test metrics bằng test_ds.
WEIGHTED_TEST_RESULTS_PATH = Path("reports/custom_cnn_v1_weighted_test_results.json")

weighted_test_results_raw = evaluation_model.evaluate(test_ds, verbose=1, return_dict=True)
weighted_test_results = {
    metric_name: float(metric_value)
    for metric_name, metric_value in weighted_test_results_raw.items()
}

WEIGHTED_TEST_RESULTS_PATH.write_text(
    json.dumps(weighted_test_results, indent=2),
    encoding="utf-8",
)

print(json.dumps(weighted_test_results, indent=2))
print(f"Saved weighted test results to: {WEIGHTED_TEST_RESULTS_PATH}")


### 8.3. Predict toàn bộ test set

> **Phụ trách:** Đinh Văn Phạm Việt (20235870) - Sinh xác suất dự đoán cho từng ảnh test.

Cell này dùng trực tiếp `x_test` để tạo unweighted metrics và phân tích lỗi.


In [ ]:
# Phụ trách: Đinh Văn Phạm Việt (20235870) - Predict toàn bộ test set để tính unweighted metrics.
y_test_proba = evaluation_model.predict(x_test, batch_size=BATCH_SIZE, verbose=1)
y_test_pred = np.argmax(y_test_proba, axis=1).astype("int64")
y_test_confidence = np.max(y_test_proba, axis=1)
softmax_row_sums = np.sum(y_test_proba, axis=1)

if y_test_proba.shape != (len(x_test), NUM_CLASSES):
    raise ValueError(f"Prediction shape mismatch: {y_test_proba.shape}")
if y_test_pred.shape != y_test.shape:
    raise ValueError(f"Predicted label shape mismatch: {y_test_pred.shape} != {y_test.shape}")
if int(y_test_pred.min()) < 0 or int(y_test_pred.max()) >= NUM_CLASSES:
    raise ValueError("Predicted labels are outside valid class range")
if not np.allclose(softmax_row_sums, np.ones_like(softmax_row_sums), atol=1e-5):
    raise ValueError("Softmax probabilities do not sum to 1")

prediction_validation_summary = {
    "prediction_shape": list(y_test_proba.shape),
    "predicted_label_shape": list(y_test_pred.shape),
    "confidence_min": float(y_test_confidence.min()),
    "confidence_mean": float(y_test_confidence.mean()),
    "confidence_max": float(y_test_confidence.max()),
    "softmax_sum_min": float(softmax_row_sums.min()),
    "softmax_sum_max": float(softmax_row_sums.max()),
    "validation_passed": True,
}

print(json.dumps(prediction_validation_summary, indent=2))


### 8.4. Tính unweighted test metrics

> **Phụ trách:** Đinh Văn Phạm Việt (20235870) - Tính metric khách quan không áp dụng sample weight.

Các metric macro rất quan trọng vì FER2013-enhanced bị mất cân bằng lớp.


In [ ]:
# Phụ trách: Đinh Văn Phạm Việt (20235870) - Tính unweighted metrics cho báo cáo test chính.
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    top_k_accuracy_score,
)

UNWEIGHTED_TEST_METRICS_PATH = Path("reports/custom_cnn_v1_unweighted_test_metrics.json")

unweighted_test_metrics = {
    "accuracy": float(accuracy_score(y_test, y_test_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_test, y_test_pred)),
    "macro_precision": float(precision_score(y_test, y_test_pred, average="macro", zero_division=0)),
    "macro_recall": float(recall_score(y_test, y_test_pred, average="macro", zero_division=0)),
    "macro_f1": float(f1_score(y_test, y_test_pred, average="macro", zero_division=0)),
    "weighted_f1": float(f1_score(y_test, y_test_pred, average="weighted", zero_division=0)),
    "top_2_accuracy": float(top_k_accuracy_score(y_test, y_test_proba, k=2, labels=np.arange(NUM_CLASSES))),
    "num_test_samples": int(len(y_test)),
}

UNWEIGHTED_TEST_METRICS_PATH.write_text(
    json.dumps(unweighted_test_metrics, indent=2),
    encoding="utf-8",
)

print(json.dumps(unweighted_test_metrics, indent=2))
print(f"Saved unweighted test metrics to: {UNWEIGHTED_TEST_METRICS_PATH}")


### 8.5. Classification report theo từng class

> **Phụ trách:** Đinh Văn Phạm Việt (20235870) - Phân tích precision, recall và F1-score theo từng cảm xúc.

Report này giúp nhóm biết class nào model học tốt và class nào còn yếu.


In [ ]:
# Phụ trách: Đinh Văn Phạm Việt (20235870) - Lưu classification report dạng JSON và CSV.
from sklearn.metrics import classification_report
import pandas as pd

CLASSIFICATION_REPORT_JSON_PATH = Path("reports/custom_cnn_v1_classification_report.json")
CLASSIFICATION_REPORT_CSV_PATH = Path("reports/custom_cnn_v1_classification_report.csv")

target_names = [label_mapping[index] for index in range(NUM_CLASSES)]
classification_report_dict = classification_report(
    y_test,
    y_test_pred,
    labels=np.arange(NUM_CLASSES),
    target_names=target_names,
    output_dict=True,
    zero_division=0,
)

CLASSIFICATION_REPORT_JSON_PATH.write_text(
    json.dumps(classification_report_dict, indent=2),
    encoding="utf-8",
)

classification_report_df = pd.DataFrame(classification_report_dict).transpose()
classification_report_df.to_csv(CLASSIFICATION_REPORT_CSV_PATH, index=True)

print(classification_report_df)
print(f"Saved classification report JSON to: {CLASSIFICATION_REPORT_JSON_PATH}")
print(f"Saved classification report CSV to: {CLASSIFICATION_REPORT_CSV_PATH}")


### 8.6. Confusion matrix

> **Phụ trách:** Đinh Văn Phạm Việt (20235870) - Trực quan hóa các lỗi nhầm lẫn giữa các cảm xúc.

Nhóm lưu cả raw count và normalized confusion matrix. Bản normalized giúp đọc tốt hơn khi dữ liệu lệch lớp.


In [ ]:
# Phụ trách: Đinh Văn Phạm Việt (20235870) - Vẽ confusion matrix raw count và normalized.
from sklearn.metrics import confusion_matrix

CONFUSION_MATRIX_RAW_PATH = Path("reports/custom_cnn_v1_confusion_matrix.png")
CONFUSION_MATRIX_NORMALIZED_PATH = Path("reports/custom_cnn_v1_confusion_matrix_normalized.png")
CONFUSION_MATRIX_JSON_PATH = Path("reports/custom_cnn_v1_confusion_matrix.json")

cm = confusion_matrix(y_test, y_test_pred, labels=np.arange(NUM_CLASSES))
cm_normalized = confusion_matrix(y_test, y_test_pred, labels=np.arange(NUM_CLASSES), normalize="true")

confusion_matrix_payload = {
    "labels": target_names,
    "raw_counts": cm.tolist(),
    "normalized_by_true_label": cm_normalized.tolist(),
}
CONFUSION_MATRIX_JSON_PATH.write_text(
    json.dumps(confusion_matrix_payload, indent=2),
    encoding="utf-8",
)

def plot_confusion_matrix(matrix, title, path, value_format):
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(matrix, cmap="Blues")
    ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks(np.arange(NUM_CLASSES))
    ax.set_yticks(np.arange(NUM_CLASSES))
    ax.set_xticklabels(target_names, rotation=45, ha="right")
    ax.set_yticklabels(target_names)

    threshold = matrix.max() / 2.0 if matrix.size else 0.0
    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            value = matrix[row, col]
            text_color = "white" if value > threshold else "black"
            ax.text(col, row, format(value, value_format), ha="center", va="center", color=text_color, fontsize=8)

    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()

plot_confusion_matrix(cm, "Confusion Matrix - Raw Counts", CONFUSION_MATRIX_RAW_PATH, "d")
plot_confusion_matrix(cm_normalized, "Confusion Matrix - Normalized by True Label", CONFUSION_MATRIX_NORMALIZED_PATH, ".2f")

print(f"Saved raw confusion matrix to: {CONFUSION_MATRIX_RAW_PATH}")
print(f"Saved normalized confusion matrix to: {CONFUSION_MATRIX_NORMALIZED_PATH}")
print(f"Saved confusion matrix JSON to: {CONFUSION_MATRIX_JSON_PATH}")


### 8.7. Phân tích các ảnh dự đoán sai

> **Phụ trách:** Đinh Văn Phạm Việt (20235870) - Lưu ví dụ lỗi sai để giải thích trong báo cáo.

Cell này chọn các ảnh sai có confidence cao nhất, vì đây là các trường hợp model tự tin nhưng vẫn nhầm.


In [ ]:
# Phụ trách: Đinh Văn Phạm Việt (20235870) - Lưu ví dụ ảnh test bị dự đoán sai.
MISCLASSIFIED_EXAMPLES_PATH = Path("reports/custom_cnn_v1_misclassified_examples.png")
MISCLASSIFIED_EXAMPLES_JSON_PATH = Path("reports/custom_cnn_v1_misclassified_examples.json")

incorrect_indices = np.where(y_test_pred != y_test)[0]
num_error_examples = min(16, len(incorrect_indices))

if num_error_examples > 0:
    sorted_error_indices = incorrect_indices[np.argsort(y_test_confidence[incorrect_indices])[::-1]]
    selected_error_indices = sorted_error_indices[:num_error_examples]
else:
    selected_error_indices = np.array([], dtype="int64")

misclassified_examples = []
for index in selected_error_indices:
    probabilities = y_test_proba[index]
    top2_indices = np.argsort(probabilities)[-2:][::-1]
    misclassified_examples.append({
        "index": int(index),
        "true_label_id": int(y_test[index]),
        "true_label": label_mapping[int(y_test[index])],
        "predicted_label_id": int(y_test_pred[index]),
        "predicted_label": label_mapping[int(y_test_pred[index])],
        "confidence": float(y_test_confidence[index]),
        "top_2_labels": [label_mapping[int(class_id)] for class_id in top2_indices],
        "top_2_probabilities": [float(probabilities[class_id]) for class_id in top2_indices],
    })

MISCLASSIFIED_EXAMPLES_JSON_PATH.write_text(
    json.dumps(misclassified_examples, indent=2),
    encoding="utf-8",
)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
axes = axes.flatten()
for axis_index, ax in enumerate(axes):
    ax.axis("off")
    if axis_index < len(selected_error_indices):
        sample_index = int(selected_error_indices[axis_index])
        true_name = label_mapping[int(y_test[sample_index])]
        pred_name = label_mapping[int(y_test_pred[sample_index])]
        confidence = float(y_test_confidence[sample_index])
        ax.imshow(x_test[sample_index].squeeze(), cmap="gray", vmin=0.0, vmax=1.0)
        ax.set_title(f"T: {true_name}\nP: {pred_name} ({confidence:.2f})", fontsize=9)

plt.tight_layout()
plt.savefig(MISCLASSIFIED_EXAMPLES_PATH, dpi=160, bbox_inches="tight")
plt.show()

print(f"Misclassified samples: {len(incorrect_indices)} / {len(y_test)}")
print(f"Saved misclassified examples image to: {MISCLASSIFIED_EXAMPLES_PATH}")
print(f"Saved misclassified examples JSON to: {MISCLASSIFIED_EXAMPLES_JSON_PATH}")


### 8.8. Confidence analysis

> **Phụ trách:** Đinh Văn Phạm Việt (20235870) - Kiểm tra model tự tin như thế nào khi đúng và khi sai.

Nếu confidence của mẫu sai quá cao, model có dấu hiệu overconfident. Nếu confidence mẫu sai thấp hơn mẫu đúng, model đang phản ánh độ khó tương đối hợp lý.


In [ ]:
# Phụ trách: Đinh Văn Phạm Việt (20235870) - Phân tích confidence của dự đoán đúng/sai.
CONFIDENCE_SUMMARY_PATH = Path("reports/custom_cnn_v1_confidence_summary.json")
CONFIDENCE_HISTOGRAM_PATH = Path("reports/custom_cnn_v1_confidence_histogram.png")

correct_mask = y_test_pred == y_test
incorrect_mask = ~correct_mask
correct_confidence = y_test_confidence[correct_mask]
incorrect_confidence = y_test_confidence[incorrect_mask]

confidence_summary = {
    "num_correct": int(correct_mask.sum()),
    "num_incorrect": int(incorrect_mask.sum()),
    "overall_confidence_mean": float(y_test_confidence.mean()),
    "correct_confidence_mean": float(correct_confidence.mean()) if len(correct_confidence) else None,
    "incorrect_confidence_mean": float(incorrect_confidence.mean()) if len(incorrect_confidence) else None,
    "correct_confidence_median": float(np.median(correct_confidence)) if len(correct_confidence) else None,
    "incorrect_confidence_median": float(np.median(incorrect_confidence)) if len(incorrect_confidence) else None,
}

CONFIDENCE_SUMMARY_PATH.write_text(
    json.dumps(confidence_summary, indent=2),
    encoding="utf-8",
)

plt.figure(figsize=(8, 5))
plt.hist(correct_confidence, bins=20, alpha=0.65, label="Correct", density=True)
plt.hist(incorrect_confidence, bins=20, alpha=0.65, label="Incorrect", density=True)
plt.xlabel("Prediction confidence")
plt.ylabel("Density")
plt.title("Confidence Distribution: Correct vs Incorrect Predictions")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(CONFIDENCE_HISTOGRAM_PATH, dpi=160, bbox_inches="tight")
plt.show()

print(json.dumps(confidence_summary, indent=2))
print(f"Saved confidence summary to: {CONFIDENCE_SUMMARY_PATH}")
print(f"Saved confidence histogram to: {CONFIDENCE_HISTOGRAM_PATH}")


### 8.9. Lưu evaluation summary

> **Phụ trách:** Đinh Văn Phạm Việt (20235870) - Tổng hợp toàn bộ kết quả đánh giá test set.

Summary này là file chính để dùng khi viết báo cáo phần đánh giá model.


In [ ]:
# Phụ trách: Đinh Văn Phạm Việt (20235870) - Lưu summary tổng hợp Step 8.
EVALUATION_SUMMARY_PATH = Path("reports/custom_cnn_v1_evaluation_summary.json")

per_class_rows = []
for label_name in target_names:
    metrics = classification_report_dict[label_name]
    per_class_rows.append({
        "label": label_name,
        "precision": float(metrics["precision"]),
        "recall": float(metrics["recall"]),
        "f1_score": float(metrics["f1-score"]),
        "support": int(metrics["support"]),
    })

best_class_by_f1 = max(per_class_rows, key=lambda item: item["f1_score"])
worst_class_by_f1 = min(per_class_rows, key=lambda item: item["f1_score"])

predicted_class_counts = {
    label_mapping[class_id]: int(count)
    for class_id, count in zip(*np.unique(y_test_pred, return_counts=True))
}
true_class_counts = {
    label_mapping[class_id]: int(count)
    for class_id, count in zip(*np.unique(y_test, return_counts=True))
}

evaluation_summary = {
    "model_path": str(EVALUATION_MODEL_PATH),
    "num_test_samples": int(len(y_test)),
    "num_correct": int(correct_mask.sum()),
    "num_incorrect": int(incorrect_mask.sum()),
    "weighted_test_results": weighted_test_results,
    "unweighted_test_metrics": unweighted_test_metrics,
    "best_class_by_f1": best_class_by_f1,
    "worst_class_by_f1": worst_class_by_f1,
    "true_class_counts": true_class_counts,
    "predicted_class_counts": predicted_class_counts,
    "confidence_summary": confidence_summary,
    "prediction_validation": prediction_validation_summary,
    "generated_files": [
        str(WEIGHTED_TEST_RESULTS_PATH),
        str(UNWEIGHTED_TEST_METRICS_PATH),
        str(CLASSIFICATION_REPORT_JSON_PATH),
        str(CLASSIFICATION_REPORT_CSV_PATH),
        str(CONFUSION_MATRIX_JSON_PATH),
        str(CONFUSION_MATRIX_RAW_PATH),
        str(CONFUSION_MATRIX_NORMALIZED_PATH),
        str(MISCLASSIFIED_EXAMPLES_PATH),
        str(MISCLASSIFIED_EXAMPLES_JSON_PATH),
        str(CONFIDENCE_SUMMARY_PATH),
        str(CONFIDENCE_HISTOGRAM_PATH),
        str(EVALUATION_SUMMARY_PATH),
    ],
}

EVALUATION_SUMMARY_PATH.write_text(
    json.dumps(evaluation_summary, indent=2),
    encoding="utf-8",
)

print(json.dumps(evaluation_summary, indent=2))
print(f"Saved evaluation summary to: {EVALUATION_SUMMARY_PATH}")


## Kết luận Step 8

> **Phụ trách:** Đinh Văn Phạm Việt (20235870) - Tổng hợp đánh giá model trên test set.  
> **Phối hợp:** Đinh Hữu Nhật Minh (20235778) - Dùng kết quả đánh giá để chọn model tích hợp demo/inference.

Sau Step 8, thư mục `reports/` cần có thêm:

- `custom_cnn_v1_weighted_test_results.json`
- `custom_cnn_v1_unweighted_test_metrics.json`
- `custom_cnn_v1_classification_report.json`
- `custom_cnn_v1_classification_report.csv`
- `custom_cnn_v1_confusion_matrix.json`
- `custom_cnn_v1_confusion_matrix.png`
- `custom_cnn_v1_confusion_matrix_normalized.png`
- `custom_cnn_v1_misclassified_examples.png`
- `custom_cnn_v1_misclassified_examples.json`
- `custom_cnn_v1_confidence_summary.json`
- `custom_cnn_v1_confidence_histogram.png`
- `custom_cnn_v1_evaluation_summary.json`

Step 8 không train thêm model. Nếu cần cải thiện model, nhóm sẽ dựa trên kết quả phân tích lỗi để lập bước thử nghiệm riêng sau.
